In [6]:
# %% [markdown]
# # Evaluation of Hybrid Recommender System (NCF + FP-Growth) with Dynamic Alpha
# Hybrid Score = alpha(u, c) * NCF(u, i) + (1 - alpha(u, c)) * Rule(c, i)

# %% [1] IMPORTS & SETUP
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import math
from pathlib import Path

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Paths (assume running from notebooks/)
MODEL_PATH = Path("../models/ncf_model.pt")
RULES_PATH = Path("../data/rules.csv")
TEST_PATH = Path("test.csv")
FULL_DATA_PATH = Path("../data/user_item_dl.csv")

K = 10

# %% [2] MODEL ARCHITECTURE (MUST MATCH TRAINING)
class NCF(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=64):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, user, item):
        u_emb = self.user_embedding(user)
        i_emb = self.item_embedding(item)
        x = torch.cat([u_emb, i_emb], dim=1)
        return self.fc(x).squeeze()

# %% [3] LOAD MODEL & MAPPINGS
print("\n--- Loading NCF Model ---")

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

user2idx = checkpoint["user2idx"]
item2idx = checkpoint["item2idx"]
idx2user = {v: k for k, v in user2idx.items()}

n_users = len(user2idx)
n_items = len(item2idx)

model = NCF(n_users, n_items).to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

print(f"Model loaded: {n_users} users, {n_items} items")

# %% [4] LOAD RULES (FP-GROWTH)
print("\n--- Loading Rules ---")

rules_df = pd.read_csv(RULES_PATH)
rule_lookup = {}

for _, row in rules_df.iterrows():
    ant = str(row["antecedent"]).replace("frozenset({", "").replace("})", "").replace("'", "")
    cons = str(row["consequent"]).replace("frozenset({", "").replace("})", "").replace("'", "")
    conf = float(row["confidence"])

    ants = [x.strip().lower() for x in ant.split(",")]
    cons = [x.strip().lower() for x in cons.split(",")]

    for a in ants:
        if a not in rule_lookup:
            rule_lookup[a] = {}
        for c in cons:
            if c in item2idx:
                c_idx = item2idx[c]
                rule_lookup[a][c_idx] = max(rule_lookup[a].get(c_idx, 0.0), conf)

print(f"Rules loaded. Antecedents: {len(rule_lookup)}")

# %% [5] LOAD FULL DATA FOR CONTEXT & INTERACTION COUNT
print("\n--- Loading User History (Train Context) ---")

full_df = pd.read_csv(FULL_DATA_PATH)
full_df["timestamp"] = pd.to_datetime(full_df["timestamp"])
full_df = full_df.sort_values(["user_id", "timestamp"])

grouped_history = full_df.groupby("user_id")["item_id"].apply(list)

# Context = second-to-last item (Leave-One-Last-Basket)
user_context_map = {}
user_interaction_count = {}

for uid, items in grouped_history.items():
    user_interaction_count[uid] = len(items)
    if len(items) >= 2:
        user_context_map[uid] = str(items[-2]).lower().strip()
    else:
        user_context_map[uid] = None

print(f"Context extracted for {len(user_context_map)} users")

# %% [6] LOAD TEST DATA
print("\n--- Loading Test Data ---")

test_df = pd.read_csv(TEST_PATH)
test_df["u_idx"] = test_df["user_id"].map(user2idx)
test_df["i_idx"] = test_df["item_id"].map(item2idx)
test_df = test_df.dropna(subset=["u_idx", "i_idx"])
test_df["u_idx"] = test_df["u_idx"].astype(int)
test_df["i_idx"] = test_df["i_idx"].astype(int)

ground_truth = test_df.groupby("u_idx")["i_idx"].apply(set).to_dict()

print(f"Test users: {len(ground_truth)}")

# %% [7] METRIC FUNCTIONS
def ndcg_at_k(rank_list, gt_set):
    dcg = 0.0
    for i, item in enumerate(rank_list):
        if item in gt_set:
            dcg += 1.0 / math.log2(i + 2)

    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(gt_set), len(rank_list))))
    return dcg / idcg if idcg > 0 else 0.0


def average_precision(rank_list, gt_set):
    hits = 0
    score = 0.0
    for i, item in enumerate(rank_list):
        if item in gt_set:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(gt_set), len(rank_list)) if hits > 0 else 0.0

# %% [8] DYNAMIC ALPHA FUNCTION
def compute_dynamic_alpha(n_interactions, u_emb, c_emb_idx, item_embedding_layer):
    # User confidence
    c_user = min(1.0, math.log1p(n_interactions) / math.log1p(50))

    # Alignment
    if c_emb_idx is not None:
        v_u = u_emb.detach().cpu().numpy()
        v_c = item_embedding_layer(
            torch.tensor(c_emb_idx).to(DEVICE)
        ).detach().cpu().numpy()

        dot = np.dot(v_u, v_c)
        norm_u = np.linalg.norm(v_u)
        norm_c = np.linalg.norm(v_c)
        s_align = dot / (norm_u * norm_c) if norm_u > 0 and norm_c > 0 else 0.0
    else:
        s_align = 0.0

    alpha = 0.2 + (0.6 * c_user) + (0.2 * s_align)
    return max(0.1, min(alpha, 0.9))

# %% [5] HYBRID EVALUATION LOOP (STRATEGY: GIVEN-1-PREDICT-REST)
print(f"\n--- Running Hybrid Evaluation (Given-1-Predict-Rest) | K={K}, Alpha=Dynamic ---")

hits_k = 0
ndcg_sum = 0
map_sum = 0
total_users = 0
covered_items = set()
# Đảm bảo idx2item đã tồn tại
if 'idx2item' not in locals():
    idx2item = {v: k for k, v in item2idx.items()}
# Pre-compute Item Tensor for fast NCF
all_items = torch.arange(n_items).to(DEVICE)

with torch.no_grad():
    for u_idx, gt_set in tqdm(ground_truth.items()):
        
        # --- STRATEGY: GIVEN-1-PREDICT-REST ---
        # Lấy danh sách item trong Test Basket
        gt_list = list(gt_set)
        
        context_idx = None
        target_set = gt_set # Mặc định là dự đoán tất cả
        
        # Nếu giỏ hàng Test có ít nhất 2 món
        # Ta lấy món đầu tiên làm "Context" (Given), dự đoán các món còn lại (Rest)
        if len(gt_list) >= 2:
            context_idx = gt_list[0]        # Món làm mồi
            target_set = set(gt_list[1:])   # Tập cần dự đoán (Ground Truth mới)
        else:
            # Nếu chỉ có 1 món, không thể tách Context trong phiên được
            # Fallback về NCF thuần túy (Context = None)
            context_idx = None
            target_set = gt_set

        # 1. NCF Prediction 
        # (Vẫn tính cho toàn bộ user như bình thường)
        u_tensor = torch.tensor([u_idx] * n_items).to(DEVICE)
        u_emb_vector = model.user_embedding(torch.tensor(u_idx).to(DEVICE))
        
        ncf_scores = model(u_tensor, all_items).cpu().numpy()
        
        # 2. Rule Boosting (Dựa trên Context "Given-1")
        rule_score_vector = np.zeros(n_items)
        
        if context_idx is not None:
            # Lấy tên item từ index để tra cứu trong Rule Dictionary
            # (Lưu ý: idx2item map ra chuỗi gốc, cần clean giống lúc load rules)
            context_item_str = str(idx2item[context_idx]).lower().strip()
            
            matches = rule_lookup.get(context_item_str, {})
            
            # matches: {target_idx: confidence}
            for t_idx, conf in matches.items():
                rule_score_vector[t_idx] = conf
                
        # 3. Dynamic Alpha Calculation
        # Tính lại Alpha dựa trên độ tương đồng giữa User (Lịch sử) và Context (Món vừa lấy ra)
        # Nếu context_idx là None thì s_align = 0
        
        # Helper logic tính alpha nhanh
        c_user = min(1.0, math.log1p(len(gt_set) + 5) / math.log1p(50)) # estimation
        s_align = 0.0
        
        if context_idx is not None:
            v_u = u_emb_vector.cpu().numpy()
            v_c = model.item_embedding(torch.tensor(context_idx).to(DEVICE)).cpu().numpy()
            norm_u = np.linalg.norm(v_u)
            norm_c = np.linalg.norm(v_c)
            if norm_u > 0 and norm_c > 0:
                s_align = np.dot(v_u, v_c) / (norm_u * norm_c)
        
        alpha_val = 0.2 + (0.6 * c_user) + (0.2 * s_align)
        alpha_val = max(0.1, min(alpha_val, 0.9))

        # 4. Hybrid Fusion
        hybrid_scores = (ncf_scores * alpha_val) + (rule_score_vector * (1.0 - alpha_val))
        
        # FILTER: Không được gợi ý lại chính món Context! (Nếu không sẽ bị tính là cheat)
        if context_idx is not None:
            hybrid_scores[context_idx] = -np.inf 

        # 5. Ranking
        top_k_indices = np.argsort(hybrid_scores)[::-1][:K]
        
        # 6. Metrics Calculation (So với target_set)
        if len(target_set) > 0: # Chỉ tính nếu còn gì đó để dự đoán
            total_users += 1
            
            is_hit = False
            for item in top_k_indices:
                covered_items.add(item)
                if item in target_set:
                    is_hit = True
            
            if is_hit: hits_k += 1
            ndcg_sum += ndcg_at_k(top_k_indices, target_set) # Sửa get_ndcg -> ndcg_at_k
            map_sum += average_precision(top_k_indices, target_set) # Sửa get_ap -> average_precision

# Tính lại Report
hr = hits_k / total_users if total_users > 0 else 0
map_score = map_sum / total_users if total_users > 0 else 0
ndcg_score = ndcg_sum / total_users if total_users > 0 else 0
coverage = len(covered_items) / n_items

print("\n=== FINAL HYBRID SYSTEM RESULTS (Strateg: Given-1) ===")
print(f"Precision/HitRate@{K}: {hr:.4f}")
print(f"NDCG@{K}:             {ndcg_score:.4f}")
print(f"MAP@{K}:              {map_score:.4f}")
print(f"Catalog Coverage:     {coverage:.2%}")
print("======================================================")

Device: cpu

--- Loading NCF Model ---
Model loaded: 4338 users, 3866 items

--- Loading Rules ---
Rules loaded. Antecedents: 265

--- Loading User History (Train Context) ---
Context extracted for 4338 users

--- Loading Test Data ---
Test users: 4338

--- Running Hybrid Evaluation (Given-1-Predict-Rest) | K=10, Alpha=Dynamic ---


100%|██████████| 4338/4338 [00:13<00:00, 332.63it/s]


=== FINAL HYBRID SYSTEM RESULTS (Strateg: Given-1) ===
Precision/HitRate@10: 0.2600
NDCG@10:             0.0518
MAP@10:              0.0253
Catalog Coverage:     20.36%
